# 10 Observed-Market Deterministic Forecast

This notebook is the canonical observed-market quarter-hour forecast interpretation layer.

Current scope:

- run or load the audited quarter-hour deterministic `D..D+4` forecast artifact;
- inspect the observed-target-only split policy and coverage;
- compare the repeated-hourly structural benchmark with the mixed-frequency quarter-hour models;
- confirm that non-observed targets are excluded from scored truth and downstream scenario-generation metadata.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Canonical Runner

The notebook loads the latest saved observed-market deterministic artifact by default. Set `RUN_OBSERVED_DETERMINISTIC = True` only when you want to rerun the full quarter-hour deterministic forecast from inside the notebook.

In [ ]:
RUN_OBSERVED_DETERMINISTIC = False

if RUN_OBSERVED_DETERMINISTIC:
    run_dir = run_observed_market_deterministic_forecast()
else:
    run_dir = find_latest_observed_deterministic_run(config)

if run_dir is None:
    raise FileNotFoundError("No observed-market deterministic quarter-hour run exists yet. Run the canonical runner first.")

run_summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
split_summary = pd.read_csv(run_dir / "split_summary.csv")
coverage_summary = pd.read_csv(run_dir / "observed_target_coverage_summary.csv")
metrics_overall = pd.read_csv(run_dir / "metrics_overall.csv")
metrics_by_lead_day = pd.read_csv(run_dir / "metrics_by_lead_day.csv")
dm_results = pd.read_csv(run_dir / "dm_test_results.csv") if (run_dir / "dm_test_results.csv").exists() else pd.DataFrame()
predictions = pd.read_csv(run_dir / "predictions_long.csv")

display(pd.DataFrame([{"observed_market_run": str(run_dir)}]))

## Split Policy And Coverage

In [ ]:
display(split_summary)
display(coverage_summary.head(20))

## Overall Metrics

In [ ]:
display(metrics_overall.sort_values(['dataset_split', 'mae', 'model']).reset_index(drop=True))

## Lead-Day Metrics

In [ ]:
display(metrics_by_lead_day.sort_values(['dataset_split', 'lead_day', 'mae', 'model']).reset_index(drop=True))

## DM Tests

In [ ]:
display(dm_results)

## Observed-Target Policy Check

The scoring contract is only valid if non-observed targets keep `y_true = NaN`.

In [ ]:
observed_policy_check = pd.DataFrame(
    [
        {
            "rows_total": int(predictions.shape[0]),
            "non_observed_rows": int((~predictions["is_observed_target"].fillna(False).astype(bool)).sum()),
            "non_observed_rows_with_non_null_y_true": int(
                predictions.loc[
                    ~predictions["is_observed_target"].fillna(False).astype(bool),
                    "y_true",
                ].notna().sum()
            ),
        }
    ]
)
display(observed_policy_check)

## Thesis Interpretation Contract

This notebook supports three narrow claims:

- the quarter-hour deterministic forecast is leakage-safe under the audited hourly availability convention;
- the observed-market evaluation uses only observed 15-minute targets as ground truth;
- the repeated-hourly benchmark remains visible so the value of added 15-minute modelling is not assumed.